# NHF validation

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_Te(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(230, 280))
    return x.sel(idx).mean(["longitude", "latitude"])


def get_Tw(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(180, 230))
    return x.sel(idx).mean(["longitude", "latitude"])

## load data

### heat/radiation fluxes

In [ ]:
## nhf
nhf_era5 = xr.open_dataset(DATA_FP / "era5_nhf_glade.nc").drop_vars("utc_date")

## make sure latitude is ascending
nhf_era5 = nhf_era5.reindex({"latitude": nhf_era5.latitude[::-1]})

## rename flux variables
varname_dict = dict(
    MSLHF="lhf",
    MSSHF="shf",
    MSNLWRF="lw",
    MSNLWRFCS="lw_cs",
    MSNSWRF="sw",
    MSNSWRFCS="sw_cs",
)
nhf_era5 = nhf_era5.rename(varname_dict)

## net heat flux
## Note: "The ECMWF convention for vertical fluxes is positive downwards."
nhf_era5["nhf"] = nhf_era5["sw"] + nhf_era5["lw"] + nhf_era5["lhf"] + nhf_era5["shf"]

Rename to make easier

### SST

In [ ]:
## load ERSST
REF_FP = pathlib.Path("/glade/campaign/cgd/cas/asphilli/CVDP-OBS/")
# pr_ref = xr.open_dataset(REF_FP / "gpcp.mon.mean.197901-202403.nc")["precip"]
sst_ersst = xr.merge(
    [
        xr.open_dataset(REF_FP / "ersstv5.185401-202412.nc")["ssta"],
        xr.open_dataset(REF_FP / "ersstv5.185401-202412.nc")["sst"],
    ]
)

sst_ersst = sst_ersst.rename({"lon": "longitude", "lat": "latitude"})
sst_ersst["time"] = xr.date_range(start="1854-01", end="2024-12", freq="MS")
sst_ersst = sst_ersst.sel(time=nhf_era5.time)

### compute indices

In [ ]:
fns = dict(
    _3=src.utils.get_nino3,
    _34=src.utils.get_nino34,
    _4=src.utils.get_nino4,
    _e=get_Te,
    _w=get_Tw,
)

In [ ]:
idx = xr.Dataset()
for ds in [nhf_era5, sst_ersst[["ssta"]]]:
    for fn_name, fn in fns.items():
        for v in list(ds):
            idx[f"{v}{fn_name}"] = fn(ds[v])

In [ ]:
idx_ndj = idx.resample({"time": "QS-NOV"}).mean()
idx_ndj = idx_ndj.isel(time=slice(4, -4, 4))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3))

for ax, suf in zip(axs, ["w", "e"]):
    ax.scatter(idx_ndj[f"ssta_{suf}"], idx_ndj[f"nhf_{suf}"])

src.utils.set_lims(axs)
axs[1].set_yticks([])
plt.show()

In [ ]:
yr = idx_ndj.time.dt.year

In [ ]:
plt.plot(yr, idx_ndj["lhf_w"])

In [ ]:
## useful things
yr = idx_ndj.time.dt.year
ax_kwargs = dict(c="k", ls="--", lw=0.8, zorder=0.5)

for suf in ["e", "w"]:
    print(f"\nLoc: {suf}")

    fig, axs = plt.subplots(2, 5, figsize=(10, 4.5), layout="constrained")

    for j, v in enumerate(["nhf", "sw", "lhf", "lw", "shf"]):

        ## scatter plot
        axs[0, j].scatter(
            idx_ndj[f"ssta_{suf}"],
            idx_ndj[f"{v}_{suf}"] - idx_ndj[f"{v}_{suf}"].mean(),
            s=20,
        )
        axs[0, j].set_title(v)
        axs[0, j].axvline(0, **ax_kwargs)
        axs[0, j].axhline(0, **ax_kwargs)
        axs[0, j].set_xticks([-2, 0, 2])
        axs[0, j].set_xlabel("ssta")

        ## timeseries
        axs[1, j].plot(yr, idx_ndj[f"{v}_{suf}"] - idx_ndj[f"{v}_{suf}"].mean())
        axs[1, j].axhline(0, **ax_kwargs)
        axs[1, j].axvline(1979, **ax_kwargs)
        axs[1, j].set_xticks([1979, 2022])

    src.utils.set_lims(axs[0])
    src.utils.set_lims(axs[1])
    for ax in axs[:, 0]:
        ax.set_ylabel(r"W m$^{-2}$")
    for ax in axs[:, 1:].flatten():
        ax.set_yticks([])
    plt.show()

Plot SST over time

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 2.25), layout="constrained")

## timeseries
for ax, suf in zip(axs[:2], ["w", "e"]):
    ax.plot(yr, idx_ndj[f"ssta_{suf}"])
    ax.axhline(0, **ax_kwargs)
    ax.axvline(1979, **ax_kwargs)
    ax.set_xticks([1979, 2022])

## scatter relationship
axs[-1].scatter(idx_ndj[f"ssta_e"], idx_ndj[f"ssta_w"], s=15)
axs[-1].axhline(0, **ax_kwargs)
axs[-1].axvline(0, **ax_kwargs)
axs[-1].set_aspect("equal")
zz = np.linspace(-2, 2.5)
# axs[-1].plot(zz, zz, **dict(ax_kwargs, zorder=10))


## format/labels
src.utils.set_lims(axs[:2])
axs[1].set_yticks([])
axs[0].set_title(r"central SST")
axs[1].set_title(r"east SST")
axs[2].set_ylabel(r"central")
axs[2].set_xlabel(r"east")

plt.show()